In [2]:
# Importing Libraries
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
os.listdir('/content/drive/MyDrive/VSP')

['CMU Data Science for Product Mgrs 94851 - Final Assignment Fall 2025.pdf',
 'LACOSTE_Sept24 ATP.xlsx',
 'Calvin Klein_Sept24 ATP.xlsx',
 'Nike_Sept24 ATP.xlsm',
 'VSP Clustering.gdoc',
 'FinalProject-old.ipynb',
 'detailed_attributes.csv',
 'AO-BI275 DEMAND KC KP LA LS KO KS 12.17.25.xlsx',
 'final_demand.csv',
 'final_demand.gsheet',
 'VSP_RF1.ipynb',
 'VSP_RF_.ipynb',
 'XGboost_VSP_finetuned.ipynb',
 'Copy of VSP Final Presentation (1).gslides',
 'Copy of VSP Final Presentation.gslides',
 'VSP_with clusteranalysis.ipynb',
 'VSP Final Presentation.gslides',
 'descriptive-analythics.ipynb',
 'FinalProject-clustering.ipynb',
 'Final Report.gdoc']

In [4]:
# Reading Demand sheet provided by VSP
df = pd.read_excel("./drive/MyDrive/VSP/AO-BI275 DEMAND KC KP LA LS KO KS 12.17.25.xlsx")
df.head()

,Collection,Brand,Material,ID,Grid value,ID-Color,Macro Region|Orig. Req. Deliv. Month/Year,09/2023,10/2023,11/2023,12/2023,01/2024,02/2024,03/2024,04/2024,05/2024,06/2024,07/2024,08/2024,Total
0,KC,CALVIN KLEIN SUN,45073,CK20541S,5719001.0,CK20541S/57/BLACK,AMER,26.0,28.0,27.0,26.0,9.0,24.0,21.0,35.0,21.0,20.0,32.0,16.0,285.0
1,KC,CALVIN KLEIN SUN,45073,CK20541S,5719001.0,CK20541S/57/BLACK,EMEA,10.0,18.0,11.0,11.0,46.0,36.0,84.0,22.0,29.0,131.0,14.0,46.0,458.0
2,KC,CALVIN KLEIN SUN,45073,CK20541S,5719235.0,CK20541S/57/DARK TOR,AMER,33.0,39.0,14.0,16.0,28.0,15.0,16.0,33.0,68.0,47.0,24.0,21.0,354.0
3,KC,CALVIN KLEIN SUN,45073,CK20541S,5719235.0,CK20541S/57/DARK TOR,EMEA,21.0,15.0,11.0,8.0,34.0,61.0,55.0,39.0,43.0,23.0,20.0,7.0,337.0
4,KC,CALVIN KLEIN SUN,45073,CK20541S,5719605.0,CK20541S/57/CRYSTAL,AMER,11.0,21.0,15.0,17.0,12.0,8.0,11.0,11.0,15.0,7.0,19.0,12.0,159.0


In [5]:
print(len(df))
df = df.dropna(subset=["ID-Color"])
print(len(df))
df["ID-Color"].isna().sum()

1866
1464


np.int64(0)

In [6]:
# Extracting the style code, size and colour from "ID-color" column

df[["StyleCode", "Size", "ColorDescription"]] = (
    df["ID-Color"]
        .str.split("/", n=2, expand=True)
)
df.head()

,Collection,Brand,Material,ID,Grid value,ID-Color,Macro Region|Orig. Req. Deliv. Month/Year,09/2023,10/2023,11/2023,...,03/2024,04/2024,05/2024,06/2024,07/2024,08/2024,Total,StyleCode,Size,ColorDescription
0,KC,CALVIN KLEIN SUN,45073,CK20541S,5719001.0,CK20541S/57/BLACK,AMER,26.0,28.0,27.0,...,21.0,35.0,21.0,20.0,32.0,16.0,285.0,CK20541S,57,BLACK
1,KC,CALVIN KLEIN SUN,45073,CK20541S,5719001.0,CK20541S/57/BLACK,EMEA,10.0,18.0,11.0,...,84.0,22.0,29.0,131.0,14.0,46.0,458.0,CK20541S,57,BLACK
2,KC,CALVIN KLEIN SUN,45073,CK20541S,5719235.0,CK20541S/57/DARK TOR,AMER,33.0,39.0,14.0,...,16.0,33.0,68.0,47.0,24.0,21.0,354.0,CK20541S,57,DARK TOR
3,KC,CALVIN KLEIN SUN,45073,CK20541S,5719235.0,CK20541S/57/DARK TOR,EMEA,21.0,15.0,11.0,...,55.0,39.0,43.0,23.0,20.0,7.0,337.0,CK20541S,57,DARK TOR
4,KC,CALVIN KLEIN SUN,45073,CK20541S,5719605.0,CK20541S/57/CRYSTAL,AMER,11.0,21.0,15.0,...,11.0,11.0,15.0,7.0,19.0,12.0,159.0,CK20541S,57,CRYSTAL


In [7]:
# Reading CSV with descriptor columns CSV for all the eyewear in the demand sheet
# detailed_attributes.csv was made using Claude where instructions were provided to scrape from eyeconic website and amazon and the rest was filled with assumptions mentioned in the report

df_attr = pd.read_csv("./drive/MyDrive/VSP/detailed_attributes.csv")
df_attr.head()

,StyleCode,FrameConstruction,FrameShape,RXAble,Gender,SunOptical,Material1,Material2,USRetailPrice,USWholesalePrice
0,CK19119,Full Rim,Square,True,Female,Optical,Acetate,Acetate,223.0,111.5
1,CK19569,Full Rim,Round,True,Unisex,Optical,Acetate,Acetate,295.0,147.5
2,CK19573,Full Rim,Rectangle,True,Male,Optical,Acetate,Acetate,180.0,90.0
3,CK20527,Full Rim,Square,True,Female,Optical,Acetate,Acetate,158.0,79.0
4,CK20531,Full Rim,Rectangle,True,Male,Optical,Acetate,Acetate,126.0,63.0


In [13]:
# Merging VSP's demand sheet with our detailed attributes sheet

df_merged = df.merge(
    df_attr,
    on="StyleCode",
    how="left"
)
df_merged

,Collection,Brand,Material,ID,Grid value,ID-Color,Macro Region|Orig. Req. Deliv. Month/Year,09/2023,10/2023,11/2023,...,ColorDescription,FrameConstruction,FrameShape,RXAble,Gender,SunOptical,Material1,Material2,USRetailPrice,USWholesalePrice
0,KC,CALVIN KLEIN SUN,45073,CK20541S,5719001.0,CK20541S/57/BLACK,AMER,26.0,28.0,27.0,...,BLACK,Full Rim,Round,True,Unisex,Sun,Acetate,Acetate,190.0,95.0
1,KC,CALVIN KLEIN SUN,45073,CK20541S,5719001.0,CK20541S/57/BLACK,EMEA,10.0,18.0,11.0,...,BLACK,Full Rim,Round,True,Unisex,Sun,Acetate,Acetate,190.0,95.0
2,KC,CALVIN KLEIN SUN,45073,CK20541S,5719235.0,CK20541S/57/DARK TOR,AMER,33.0,39.0,14.0,...,DARK TOR,Full Rim,Round,True,Unisex,Sun,Acetate,Acetate,190.0,95.0
3,KC,CALVIN KLEIN SUN,45073,CK20541S,5719235.0,CK20541S/57/DARK TOR,EMEA,21.0,15.0,11.0,...,DARK TOR,Full Rim,Round,True,Unisex,Sun,Acetate,Acetate,190.0,95.0
4,KC,CALVIN KLEIN SUN,45073,CK20541S,5719605.0,CK20541S/57/CRYSTAL,AMER,11.0,21.0,15.0,...,CRYSTAL,Full Rim,Round,True,Unisex,Sun,Acetate,Acetate,190.0,95.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1459,LS,LACOSTE SUNS,L995S,L995S,5318002.0,L995S/53/MATTE BLACK,EMEA,102.0,102.0,110.0,...,MATTE BLACK,Full Rim,Rectangle,True,Male,Sun,Acetate,Acetate,185.0,92.5
1460,LS,LACOSTE SUNS,L995S,L995S,5318214.0,L995S/53/HAVANA,AMER,1.0,2.0,NaN,...,HAVANA,Full Rim,Rectangle,True,Male,Sun,Acetate,Acetate,185.0,92.5
1461,LS,LACOSTE SUNS,L995S,L995S,5318214.0,L995S/53/HAVANA,EMEA,26.0,81.0,93.0,...,HAVANA,Full Rim,Rectangle,True,Male,Sun,Acetate,Acetate,185.0,92.5
1462,LS,LACOSTE SUNS,L995S,L995S,5318401.0,L995S/53/MATTE BLUE,AMER,7.0,3.0,2.0,...,MATTE BLUE,Full Rim,Rectangle,True,Male,Sun,Acetate,Acetate,185.0,92.5


In [9]:
print(df_merged['StyleCode'].nunique())

220


In [10]:
df_merged = df_merged.applymap(
    lambda x: x.upper() if isinstance(x, str) else x
)

/tmp/ipython-input-1289/4150741473.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_merged = df_merged.applymap(


In [11]:
# Dropping repetitive columns so data is cleaner to work with

df_merged = df_merged.drop(columns=['Material','ID','Grid value','ID-Color'])
df_merged

,Collection,Brand,Macro Region|Orig. Req. Deliv. Month/Year,09/2023,10/2023,11/2023,12/2023,01/2024,02/2024,03/2024,...,ColorDescription,FrameConstruction,FrameShape,RXAble,Gender,SunOptical,Material1,Material2,USRetailPrice,USWholesalePrice
0,KC,CALVIN KLEIN SUN,AMER,26.0,28.0,27.0,26.0,9.0,24.0,21.0,...,BLACK,FULL RIM,ROUND,True,UNISEX,SUN,ACETATE,ACETATE,190.0,95.0
1,KC,CALVIN KLEIN SUN,EMEA,10.0,18.0,11.0,11.0,46.0,36.0,84.0,...,BLACK,FULL RIM,ROUND,True,UNISEX,SUN,ACETATE,ACETATE,190.0,95.0
2,KC,CALVIN KLEIN SUN,AMER,33.0,39.0,14.0,16.0,28.0,15.0,16.0,...,DARK TOR,FULL RIM,ROUND,True,UNISEX,SUN,ACETATE,ACETATE,190.0,95.0
3,KC,CALVIN KLEIN SUN,EMEA,21.0,15.0,11.0,8.0,34.0,61.0,55.0,...,DARK TOR,FULL RIM,ROUND,True,UNISEX,SUN,ACETATE,ACETATE,190.0,95.0
4,KC,CALVIN KLEIN SUN,AMER,11.0,21.0,15.0,17.0,12.0,8.0,11.0,...,CRYSTAL,FULL RIM,ROUND,True,UNISEX,SUN,ACETATE,ACETATE,190.0,95.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1459,LS,LACOSTE SUNS,EMEA,102.0,102.0,110.0,70.0,176.0,266.0,295.0,...,MATTE BLACK,FULL RIM,RECTANGLE,True,MALE,SUN,ACETATE,ACETATE,185.0,92.5
1460,LS,LACOSTE SUNS,AMER,1.0,2.0,NaN,1.0,NaN,NaN,3.0,...,HAVANA,FULL RIM,RECTANGLE,True,MALE,SUN,ACETATE,ACETATE,185.0,92.5
1461,LS,LACOSTE SUNS,EMEA,26.0,81.0,93.0,21.0,121.0,198.0,97.0,...,HAVANA,FULL RIM,RECTANGLE,True,MALE,SUN,ACETATE,ACETATE,185.0,92.5
1462,LS,LACOSTE SUNS,AMER,7.0,3.0,2.0,3.0,1.0,1.0,1.0,...,MATTE BLUE,FULL RIM,RECTANGLE,True,MALE,SUN,ACETATE,ACETATE,185.0,92.5


In [12]:
# Exporting data to use for

output_path = "/content/drive/MyDrive/VSP/final_demand.csv"
df_merged.to_csv(output_path, index=False)